<a href="https://colab.research.google.com/github/sayandeepmaity/luminator/blob/main/predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
import pandas as pd
import numpy as np

# Number of test samples you want to generate
num_samples = 100

# Base features used by Model 1 and Model 2
features = [
    'correlation', 'phase', 'amplitude', 'energy', 'spectral_centroid', 'snr',
    'tau_01', 'tau_02', 'tau_03', 'tau_04', 'tau_05',
    'tau_12', 'tau_13', 'tau_14', 'tau_15', 'tau_23', 'tau_24', 'tau_25',
    'tau_34', 'tau_35', 'tau_45',
    'corr_01', 'corr_02', 'corr_03', 'corr_04', 'corr_05',
    'corr_12', 'corr_13', 'corr_14', 'corr_15',
    'corr_23', 'corr_24', 'corr_25', 'corr_34', 'corr_35', 'corr_45',
    'snr_01', 'snr_02', 'snr_03', 'snr_04', 'snr_05',
    'snr_12', 'snr_13', 'snr_14', 'snr_15',
    'snr_23', 'snr_24', 'snr_25', 'snr_34', 'snr_35', 'snr_45'
]

# Create random but realistic values for each feature
data = {feature: np.random.rand(num_samples) * np.random.uniform(1, 10) for feature in features}

# Create the DataFrame
df = pd.DataFrame(data)

# Save to CSV with the updated file path
df.to_csv('/content/drive/MyDrive/gunshotdata/testgundata.csv', index=False)

print("✅ testgundata.csv created with shape:", df.shape)


✅ testgundata.csv created with shape: (100, 51)


In [35]:
import pandas as pd
import joblib
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Load the trained models
model_gunshot = joblib.load('/content/drive/My Drive/gunshotdata/dmodel1/gunshot_classifier_model.pkl')
model_location = joblib.load('/content/drive/My Drive/gunshotdata/dmodel2/model2_tdoaselector.pkl')
model3 = joblib.load('/content/drive/My Drive/gunshotdata/dmodel3/gun_type_model.pkl')

# Load the test data
df = pd.read_csv('/content/drive/MyDrive/gunshotdata/testgundata.csv')

# Features for Model 1
required_features_model1 = ['correlation', 'phase', 'amplitude', 'energy', 'spectral_centroid', 'snr']
df_model1 = df[required_features_model1]

# Features for Model 2 (assumed)
required_features_model2 = ['amplitude', 'energy', 'spectral_centroid', 'snr',
                            'tau_01', 'tau_02', 'tau_03', 'tau_04', 'tau_05', 'tau_12', 'tau_13', 'tau_14',
                            'tau_15', 'tau_23', 'tau_24', 'tau_25', 'tau_34', 'tau_35', 'tau_45', 'corr_01',
                            'corr_02', 'corr_03', 'corr_04', 'corr_05', 'corr_12', 'corr_13', 'corr_14',
                            'corr_15', 'corr_23', 'corr_24', 'corr_25', 'corr_34', 'corr_35', 'corr_45',
                            'snr_01', 'snr_02', 'snr_03', 'snr_04', 'snr_05', 'snr_12', 'snr_13', 'snr_14',
                            'snr_15', 'snr_23', 'snr_24', 'snr_25', 'snr_34', 'snr_35', 'snr_45']

# Features for Model 3
required_features_model3 = required_features_model2  # Same as Model 2 in this case

# Store results
results = []

# Get correct feature orders
features_model3_ordered = model3.feature_names_in_
features_model2_ordered = model_location.feature_names_in_

# Loop through each sample
for idx, row in df_model1.iterrows():
    gunshot_pred = model_gunshot.predict(pd.DataFrame([row]))[0]

    if gunshot_pred == 1:
        # Extract full row
        full_row = df.loc[idx, :]

        # Model 3 (Gun Type)
        row_model3_filtered = pd.DataFrame([full_row[features_model3_ordered].values], columns=features_model3_ordered)
        gun_type_pred = model3.predict(row_model3_filtered)[0]

        # Model 2 (Location/Best Pair)
        row_model2_filtered = pd.DataFrame([full_row[features_model2_ordered].values], columns=features_model2_ordered)
        location_pred = model_location.predict(row_model2_filtered)[0]

        # Save result
        results.append({
            'sample_index': idx,
            'gunshot_detected': True,
            'gun_type': gun_type_pred,
            'location_predicted': location_pred
        })

        print(f"✅ Sample {idx}: Gunshot detected!")
        print(f"   Gun Type: {gun_type_pred}")
        print(f"   Location Prediction (Model 2): {location_pred}")

    else:
        results.append({
            'sample_index': idx,
            'gunshot_detected': False,
            'gun_type': None,
            'location_predicted': None
        })
        print(f"❌ Sample {idx}: No gunshot detected.")

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv('/content/drive/MyDrive/gunshotdata/gunshot_predictions_with_location.csv', index=False)

print("✅ All predictions saved to 'gunshot_predictions_with_location.csv'.")


✅ Sample 0: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 5
✅ Sample 1: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 14
✅ Sample 2: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 0
✅ Sample 3: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 14
✅ Sample 4: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 7
✅ Sample 5: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 9
✅ Sample 6: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 14
✅ Sample 7: Gunshot detected!
   Gun Type: Rifle
   Location Prediction (Model 2): 0
✅ Sample 8: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 4
✅ Sample 9: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 9
✅ Sample 10: Gunshot detected!
   Gun Type: Pistol
   Location Prediction (Model 2): 14
✅ Sample 11: Gunshot detected!
   Gun Type: Pistol